# 🫀 Notebook 3 — Cardiovascular Disease Dataset EDA
**Dataset**: Cardiovascular Disease Dataset  
**Source**: [Kaggle — Sulianova](https://www.kaggle.com/datasets/sulianova/cardiovascular-disease-dataset)  
**Rows**: 70,000 | **Features**: 12 (+derived BMI) | **Target**: `cardio` (used as hypertension proxy)

### Objective
EDA on the largest dataset in our pipeline. Key engineering tasks:
- Derive age in years from `age` (stored in days)
- Derive BMI from height and weight
- Analyse blood pressure distributions (systolic `ap_hi`, diastolic `ap_lo`)
- Understand lifestyle factor contributions


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

TEAL   = "#117A65"
ORANGE = "#D35400"
BLUE   = "#2D6A9F"
BG     = "#F7F9FB"
DARK   = "#1C2833"
RED    = "#C0392B"

np.random.seed(42)
print("Libraries loaded ✅")


## 1 · Load Dataset

In [ ]:
def make_cardio(n=70000):
    """Synthesise Cardiovascular Disease data (70,000 rows)."""
    neg = int(n * 0.501); pos = n - neg
    df = pd.DataFrame({
        'age':        np.concatenate([np.random.normal(19200, 3200, neg),
                                      np.random.normal(20800, 3000, pos)]).astype(int),
        'gender':     np.concatenate([np.random.choice([1,2], neg), np.random.choice([1,2], pos)]),
        'height':     np.concatenate([np.random.normal(165, 10, neg), np.random.normal(163, 10, pos)]).astype(int),
        'weight':     np.concatenate([np.random.normal(72, 14, neg), np.random.normal(79, 15, pos)]),
        'ap_hi':      np.concatenate([np.random.normal(120,15,neg), np.random.normal(135,19,pos)]).clip(60,220),
        'ap_lo':      np.concatenate([np.random.normal(79,10,neg), np.random.normal(88,13,pos)]).clip(40,140),
        'cholesterol':np.concatenate([np.random.choice([1,2,3],neg,p=[0.68,0.20,0.12]),
                                      np.random.choice([1,2,3],pos,p=[0.45,0.25,0.30])]),
        'gluc':       np.concatenate([np.random.choice([1,2,3],neg,p=[0.77,0.12,0.11]),
                                      np.random.choice([1,2,3],pos,p=[0.68,0.16,0.16])]),
        'smoke':      np.concatenate([np.random.choice([0,1],neg,p=[0.91,0.09]),
                                      np.random.choice([0,1],pos,p=[0.89,0.11])]),
        'alco':       np.concatenate([np.random.choice([0,1],neg,p=[0.94,0.06]),
                                      np.random.choice([0,1],pos,p=[0.94,0.06])]),
        'active':     np.concatenate([np.random.choice([0,1],neg,p=[0.20,0.80]),
                                      np.random.choice([0,1],pos,p=[0.26,0.74])]),
        'cardio':     np.array([0]*neg + [1]*pos)
    })
    # Feature engineering
    df['BMI'] = (df['weight'] / (df['height']/100)**2).round(2)
    df['age_years'] = (df['age'] / 365.25).round(1)
    return df.sample(frac=1, random_state=42).reset_index(drop=True)

df = make_cardio()
# df = pd.read_csv('cardio_train.csv', sep=';')
# df['BMI'] = df['weight'] / (df['height']/100)**2
# df['age_years'] = df['age'] / 365.25

print(f"Shape: {df.shape}")
df.head()


## 2 · Data Overview & Feature Engineering

In [ ]:
print("Dataset Info:")
df.info()
print("\nSynthesised BMI stats:")
print(df['BMI'].describe().round(2))
print("\nAge range:", df['age_years'].min(), '—', df['age_years'].max(), 'years')


## 3 · Class Distribution

In [ ]:
col_map = {0: TEAL, 1: ORANGE}; label_map = {0: 'No CVD', 1: 'CVD'}
fig, axes = plt.subplots(1, 2, figsize=(12, 4), facecolor=BG)
counts = df['cardio'].value_counts().sort_index()
labels = ['No CVD', 'CVD']
axes[0].bar(labels, counts.values, color=[TEAL, ORANGE], edgecolor='white', width=0.5)
for bar, v in zip(axes[0].patches, counts.values):
    axes[0].text(bar.get_x()+bar.get_width()/2, v+200, f'{v:,}',
                 ha='center', va='bottom', fontweight='bold', color=DARK)
axes[0].set_title('Class Distribution (n=70,000)', fontweight='bold'); axes[0].set_facecolor(BG)
axes[0].spines[['top','right']].set_visible(False)
axes[1].pie(counts.values, labels=labels, colors=[TEAL, ORANGE],
            autopct='%1.1f%%', startangle=90, wedgeprops=dict(edgecolor='white', linewidth=2))
axes[1].set_title('Class Proportions', fontweight='bold'); axes[1].set_facecolor(BG)
plt.tight_layout(); plt.show()


## 4 · Blood Pressure Analysis

In [ ]:
sample = df.sample(5000, random_state=42)
fig, axes = plt.subplots(1, 3, figsize=(18, 5), facecolor=BG)

for outcome, clr in col_map.items():
    sub = sample[sample['cardio']==outcome]
    axes[0].hist(sub['ap_hi'], bins=40, alpha=0.6, color=clr, label=label_map[outcome],
                 edgecolor='white', linewidth=0.3)
    axes[1].hist(sub['ap_lo'], bins=40, alpha=0.6, color=clr, label=label_map[outcome],
                 edgecolor='white', linewidth=0.3)
    axes[2].scatter(sub['ap_hi'], sub['ap_lo'], c=clr, alpha=0.15, s=8, label=label_map[outcome])

axes[0].set_title('Systolic BP (ap_hi)', fontweight='bold', color=DARK)
axes[1].set_title('Diastolic BP (ap_lo)', fontweight='bold', color=DARK)
axes[2].set_title('Systolic vs Diastolic BP', fontweight='bold', color=DARK)
axes[2].set(xlabel='Systolic BP', ylabel='Diastolic BP')

for ax in axes:
    ax.legend(fontsize=8); ax.set_facecolor(BG); ax.spines[['top','right']].set_visible(False)
fig.suptitle('Blood Pressure Distributions', fontsize=14, fontweight='bold', color=DARK)
plt.tight_layout(); plt.show()


## 5 · BMI and Age Distributions

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5), facecolor=BG)
for outcome, clr in col_map.items():
    sub = sample[sample['cardio']==outcome]
    axes[0].hist(sub['age_years'], bins=30, alpha=0.65, color=clr, label=label_map[outcome],
                 edgecolor='white', linewidth=0.4, density=True)
    axes[1].hist(sub['BMI'], bins=40, alpha=0.65, color=clr, label=label_map[outcome],
                 edgecolor='white', linewidth=0.4, density=True)
    axes[2].scatter(sub['age_years'], sub['ap_hi'], c=clr, alpha=0.15, s=8, label=label_map[outcome])

for thresh, clr_t, lbl_t in [(18.5, 'green', 'Underweight'), (25, 'orange', 'Overweight'), (30, 'red', 'Obese')]:
    axes[1].axvline(thresh, color=clr_t, linestyle='--', linewidth=1.5, label=lbl_t)

axes[0].set_title('Age Distribution', fontweight='bold', color=DARK)
axes[1].set_title('BMI Distribution', fontweight='bold', color=DARK)
axes[2].set_title('Age vs Systolic BP', fontweight='bold', color=DARK)
axes[2].set(xlabel='Age (years)', ylabel='Systolic BP')

for ax in axes:
    ax.legend(fontsize=7); ax.set_facecolor(BG); ax.spines[['top','right']].set_visible(False)
plt.tight_layout(); plt.show()


## 6 · Lifestyle Factors

In [ ]:
life_feats = ['smoke', 'alco', 'active']
life_labels = {'smoke':'Smoking', 'alco':'Alcohol', 'active':'Physically Active'}

fig, axes = plt.subplots(1, 3, figsize=(15, 4), facecolor=BG)
for ax, feat in zip(axes, life_feats):
    ct = df.groupby([feat, 'cardio']).size().unstack(fill_value=0)
    ct.plot(kind='bar', ax=ax, color=[TEAL, ORANGE], edgecolor='white', alpha=0.85, width=0.6)
    ax.set_title(life_labels[feat], fontweight='bold', color=DARK)
    ax.set_xticklabels(['No', 'Yes'], rotation=0); ax.legend(['No CVD', 'CVD'], fontsize=8)
    ax.set_facecolor(BG); ax.spines[['top','right']].set_visible(False)
fig.suptitle('Lifestyle Factors vs CVD Status', fontsize=14, fontweight='bold', color=DARK)
plt.tight_layout(); plt.show()


## 7 · Cholesterol & Glucose Levels

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), facecolor=BG)
for i, feat in enumerate(['cholesterol', 'gluc']):
    ct = df.groupby([feat, 'cardio']).size().unstack(fill_value=0)
    ct.plot(kind='bar', ax=axes[i], color=[TEAL, ORANGE], edgecolor='white', alpha=0.85, width=0.6)
    level_labels = ['Normal', 'Above normal', 'Well above normal']
    axes[i].set_xticklabels(level_labels, rotation=20, fontsize=8)
    axes[i].set_title(feat.capitalize(), fontweight='bold', color=DARK)
    axes[i].legend(['No CVD', 'CVD'], fontsize=9)
    axes[i].set_facecolor(BG); axes[i].spines[['top','right']].set_visible(False)
fig.suptitle('Cholesterol & Glucose Levels vs CVD Status', fontsize=14, fontweight='bold', color=DARK)
plt.tight_layout(); plt.show()


## 8 · Summary

| Feature | Description | Cascade Role |
|---|---|---|
| `age_years` | Age in years (derived from days) | Universal confounder |
| `BMI` | Derived from height + weight | Forwarded to downstream models |
| `ap_hi` | Systolic blood pressure | Key hypertension indicator; forwarded to heart disease |
| `ap_lo` | Diastolic blood pressure | Secondary BP signal |
| `cholesterol` | Cholesterol level (categorical) | Forwarded to heart disease model |
| `gluc` | Glucose level (categorical) | Shared with diabetes signal |
| `smoke` | Smoking status | Modifiable risk factor |
| `active` | Physical activity | Modifiable protective factor |
